# AI-Driven Automated Test Case Generation with RAG

This notebook implements a complete RAG (Retrieval Augmented Generation) system for automated test case generation.

## Features:
- Code and documentation preprocessing
- Vector embeddings for code context
- RAG-based retrieval of relevant code snippets
- LLM-powered test case generation (unit, integration, edge cases)
- Evaluation metrics (coverage, quality)

## Architecture:
1. **Data Collection**: Gather code, docs, and existing tests
2. **Preprocessing**: Parse and chunk code/docs
3. **Embedding**: Create vector embeddings
4. **Vector Store**: Store in FAISS/ChromaDB
5. **RAG Retrieval**: Fetch relevant context
6. **Test Generation**: Use LLM to generate tests
7. **Evaluation**: Measure quality and coverage

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q langchain langchain-community langchain-openai
!pip install -q chromadb faiss-cpu sentence-transformers
!pip install -q openai anthropic tiktoken
!pip install -q tree-sitter tree-sitter-languages ast-comments
!pip install -q pytest coverage radon
!pip install -q python-dotenv tqdm rich

In [ ]:
# Import required libraries
import os
import re
import ast
import json
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from collections import defaultdict

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter, Language
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Utilities
import tiktoken
from tqdm.auto import tqdm
from rich import print as rprint
from rich.console import Console
from rich.table import Table

warnings.filterwarnings('ignore')
console = Console()

## 2. Configuration

In [ ]:
@dataclass
class RAGConfig:
    """Configuration for RAG-based test generation system"""
    
    # Model configuration
    llm_model: str = "gpt-4-turbo-preview"  # or "claude-3-5-sonnet-20241022"
    embedding_model: str = "sentence-transformers/all-mpnet-base-v2"
    temperature: float = 0.7
    max_tokens: int = 2000
    
    # Chunking configuration
    chunk_size: int = 1000
    chunk_overlap: int = 200
    
    # RAG configuration
    top_k_retrieval: int = 5
    similarity_threshold: float = 0.7
    
    # Vector store
    vector_store_type: str = "faiss"  # "faiss" or "chroma"
    persist_directory: str = "./vector_store"
    
    # Test generation
    test_types: List[str] = None
    include_edge_cases: bool = True
    include_negative_tests: bool = True
    
    def __post_init__(self):
        if self.test_types is None:
            self.test_types = ["unit", "integration", "edge_case"]

# Initialize configuration
config = RAGConfig()
console.print("[bold green]Configuration initialized:[/bold green]")
console.print(config)

In [ ]:
# Set up API keys (use environment variables or .env file)
from dotenv import load_dotenv

load_dotenv()

# For OpenAI
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")

# For Anthropic (if using Claude)
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-api-key-here")

if OPENAI_API_KEY == "your-api-key-here":
    console.print("[yellow]Warning: Please set your API keys in environment variables or .env file[/yellow]")

## 3. Code Parser and Preprocessor

In [ ]:
class CodeParser:
    """Parse and extract information from source code"""
    
    def __init__(self):
        self.supported_languages = ['.py', '.js', '.java', '.cpp', '.ts']
    
    def extract_functions_python(self, code: str, file_path: str = "") -> List[Dict[str, Any]]:
        """Extract functions from Python code using AST"""
        functions = []
        
        try:
            tree = ast.parse(code)
            
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    # Extract function details
                    func_info = {
                        'name': node.name,
                        'file_path': file_path,
                        'line_start': node.lineno,
                        'line_end': node.end_lineno,
                        'args': [arg.arg for arg in node.args.args],
                        'docstring': ast.get_docstring(node) or "",
                        'is_async': isinstance(node, ast.AsyncFunctionDef),
                        'decorators': [d.id if isinstance(d, ast.Name) else str(d) for d in node.decorator_list],
                        'source': ast.get_source_segment(code, node) if hasattr(ast, 'get_source_segment') else ""
                    }
                    
                    # Extract return type if annotated
                    if node.returns:
                        func_info['return_type'] = ast.unparse(node.returns) if hasattr(ast, 'unparse') else str(node.returns)
                    
                    functions.append(func_info)
                    
        except SyntaxError as e:
            console.print(f"[red]Syntax error parsing {file_path}: {e}[/red]")
        
        return functions
    
    def extract_classes_python(self, code: str, file_path: str = "") -> List[Dict[str, Any]]:
        """Extract classes from Python code"""
        classes = []
        
        try:
            tree = ast.parse(code)
            
            for node in ast.walk(tree):
                if isinstance(node, ast.ClassDef):
                    methods = []
                    for item in node.body:
                        if isinstance(item, ast.FunctionDef):
                            methods.append(item.name)
                    
                    class_info = {
                        'name': node.name,
                        'file_path': file_path,
                        'line_start': node.lineno,
                        'line_end': node.end_lineno,
                        'bases': [ast.unparse(base) if hasattr(ast, 'unparse') else str(base) for base in node.bases],
                        'methods': methods,
                        'docstring': ast.get_docstring(node) or "",
                    }
                    
                    classes.append(class_info)
                    
        except SyntaxError as e:
            console.print(f"[red]Syntax error parsing {file_path}: {e}[/red]")
        
        return classes
    
    def get_imports(self, code: str) -> List[str]:
        """Extract import statements"""
        imports = []
        
        try:
            tree = ast.parse(code)
            for node in ast.walk(tree):
                if isinstance(node, ast.Import):
                    for alias in node.names:
                        imports.append(alias.name)
                elif isinstance(node, ast.ImportFrom):
                    module = node.module or ''
                    for alias in node.names:
                        imports.append(f"{module}.{alias.name}")
        except:
            pass
        
        return imports

# Test the parser
parser = CodeParser()
console.print("[bold green]Code parser initialized[/bold green]")

In [ ]:
class CodePreprocessor:
    """Preprocess code files for RAG system"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.parser = CodeParser()
        self.text_splitter = RecursiveCharacterTextSplitter.from_language(
            language=Language.PYTHON,
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap
        )
    
    def load_code_files(self, directory: str, extensions: List[str] = None) -> List[Dict[str, Any]]:
        """Load all code files from directory"""
        if extensions is None:
            extensions = ['.py', '.js', '.java', '.ts', '.cpp']
        
        code_files = []
        directory_path = Path(directory)
        
        for ext in extensions:
            for file_path in directory_path.rglob(f"*{ext}"):
                # Skip test files and virtual environments
                if any(skip in str(file_path) for skip in ['test_', '_test', 'venv', 'node_modules', '.git']):
                    continue
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    
                    code_files.append({
                        'path': str(file_path),
                        'content': content,
                        'extension': ext,
                        'size': len(content)
                    })
                except Exception as e:
                    console.print(f"[yellow]Error reading {file_path}: {e}[/yellow]")
        
        return code_files
    
    def create_documents(self, code_files: List[Dict[str, Any]]) -> List[Document]:
        """Create LangChain documents from code files"""
        documents = []
        
        for file_info in tqdm(code_files, desc="Processing code files"):
            content = file_info['content']
            file_path = file_info['path']
            
            # Parse code structure
            if file_info['extension'] == '.py':
                functions = self.parser.extract_functions_python(content, file_path)
                classes = self.parser.extract_classes_python(content, file_path)
                imports = self.parser.get_imports(content)
                
                # Create documents for each function
                for func in functions:
                    doc = Document(
                        page_content=f"""Function: {func['name']}
File: {file_path}
Arguments: {', '.join(func['args'])}
Docstring: {func['docstring']}

Source Code:
{func.get('source', '')}""",
                        metadata={
                            'type': 'function',
                            'name': func['name'],
                            'file_path': file_path,
                            'line_start': func['line_start'],
                            'language': 'python'
                        }
                    )
                    documents.append(doc)
                
                # Create documents for each class
                for cls in classes:
                    doc = Document(
                        page_content=f"""Class: {cls['name']}
File: {file_path}
Base Classes: {', '.join(cls['bases'])}
Methods: {', '.join(cls['methods'])}
Docstring: {cls['docstring']}""",
                        metadata={
                            'type': 'class',
                            'name': cls['name'],
                            'file_path': file_path,
                            'language': 'python'
                        }
                    )
                    documents.append(doc)
            
            # Also create chunked documents for full file context
            chunks = self.text_splitter.split_text(content)
            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk,
                    metadata={
                        'type': 'code_chunk',
                        'file_path': file_path,
                        'chunk_index': i,
                        'language': file_info['extension'][1:]
                    }
                )
                documents.append(doc)
        
        return documents

# Initialize preprocessor
preprocessor = CodePreprocessor(config)
console.print("[bold green]Code preprocessor initialized[/bold green]")

## 4. Vector Store and Embeddings

In [ ]:
class VectorStoreManager:
    """Manage vector store for code embeddings"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.embeddings = None
        self.vector_store = None
        self._initialize_embeddings()
    
    def _initialize_embeddings(self):
        """Initialize embedding model"""
        console.print(f"[cyan]Loading embedding model: {self.config.embedding_model}[/cyan]")
        
        # Use HuggingFace embeddings (free and good quality)
        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.config.embedding_model,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        
        console.print("[green]Embeddings initialized[/green]")
    
    def create_vector_store(self, documents: List[Document]) -> Any:
        """Create and populate vector store"""
        console.print(f"[cyan]Creating {self.config.vector_store_type} vector store with {len(documents)} documents[/cyan]")
        
        if self.config.vector_store_type == "faiss":
            self.vector_store = FAISS.from_documents(
                documents=documents,
                embedding=self.embeddings
            )
        elif self.config.vector_store_type == "chroma":
            self.vector_store = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                persist_directory=self.config.persist_directory
            )
        else:
            raise ValueError(f"Unsupported vector store type: {self.config.vector_store_type}")
        
        console.print("[bold green]Vector store created successfully[/bold green]")
        return self.vector_store
    
    def save_vector_store(self, path: str = None):
        """Save vector store to disk"""
        if path is None:
            path = self.config.persist_directory
        
        if self.config.vector_store_type == "faiss":
            os.makedirs(path, exist_ok=True)
            self.vector_store.save_local(path)
        elif self.config.vector_store_type == "chroma":
            self.vector_store.persist()
        
        console.print(f"[green]Vector store saved to {path}[/green]")
    
    def load_vector_store(self, path: str = None):
        """Load vector store from disk"""
        if path is None:
            path = self.config.persist_directory
        
        if self.config.vector_store_type == "faiss":
            self.vector_store = FAISS.load_local(
                path,
                self.embeddings,
                allow_dangerous_deserialization=True
            )
        elif self.config.vector_store_type == "chroma":
            self.vector_store = Chroma(
                persist_directory=path,
                embedding_function=self.embeddings
            )
        
        console.print(f"[green]Vector store loaded from {path}[/green]")
        return self.vector_store
    
    def retrieve_relevant_context(
        self, 
        query: str, 
        k: int = None,
        filter_dict: Dict = None
    ) -> List[Document]:
        """Retrieve relevant documents for a query"""
        if k is None:
            k = self.config.top_k_retrieval
        
        if filter_dict:
            results = self.vector_store.similarity_search(
                query, 
                k=k,
                filter=filter_dict
            )
        else:
            results = self.vector_store.similarity_search(query, k=k)
        
        return results

# Initialize vector store manager
vector_manager = VectorStoreManager(config)
console.print("[bold green]Vector store manager initialized[/bold green]")

## 5. RAG-Based Test Case Generator

In [ ]:
class TestCaseGenerator:
    """Generate test cases using RAG and LLM"""
    
    def __init__(self, config: RAGConfig, vector_manager: VectorStoreManager):
        self.config = config
        self.vector_manager = vector_manager
        self.llm = self._initialize_llm()
        self.prompts = self._create_prompts()
    
    def _initialize_llm(self):
        """Initialize the language model"""
        console.print(f"[cyan]Initializing LLM: {self.config.llm_model}[/cyan]")
        
        llm = ChatOpenAI(
            model=self.config.llm_model,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            api_key=OPENAI_API_KEY
        )
        
        return llm
    
    def _create_prompts(self) -> Dict[str, PromptTemplate]:
        """Create prompt templates for different test types"""
        prompts = {}
        
        # Unit test prompt
        prompts['unit'] = PromptTemplate(
            input_variables=["context", "code", "function_name"],
            template="""You are an expert software testing engineer. Generate comprehensive unit tests for the following code.

CONTEXT FROM CODEBASE:
{context}

CODE TO TEST:
{code}

FUNCTION/CLASS NAME: {function_name}

Generate unit tests that:
1. Test normal/happy path scenarios
2. Test edge cases and boundary conditions
3. Test error handling and exceptions
4. Use appropriate mocking for dependencies
5. Follow pytest best practices
6. Include clear test names and docstrings
7. Aim for high code coverage

Provide the complete test code with imports and setup:"""
        )
        
        # Integration test prompt
        prompts['integration'] = PromptTemplate(
            input_variables=["context", "code", "function_name"],
            template="""You are an expert software testing engineer. Generate integration tests for the following code.

CONTEXT FROM CODEBASE:
{context}

CODE TO TEST:
{code}

COMPONENT NAME: {function_name}

Generate integration tests that:
1. Test interactions between multiple components
2. Test data flow and state management
3. Test external dependencies and APIs
4. Include setup and teardown procedures
5. Use appropriate fixtures and test data
6. Follow pytest best practices

Provide the complete test code with imports and setup:"""
        )
        
        # Edge case prompt
        prompts['edge_case'] = PromptTemplate(
            input_variables=["context", "code", "function_name"],
            template="""You are an expert software testing engineer specializing in edge cases and corner cases.

CONTEXT FROM CODEBASE:
{context}

CODE TO TEST:
{code}

FUNCTION/CLASS NAME: {function_name}

Generate edge case tests that cover:
1. Boundary values (empty, zero, max, min)
2. Invalid inputs and type errors
3. Null/None cases
4. Concurrent access and race conditions
5. Resource exhaustion scenarios
6. Unusual but valid inputs
7. Security edge cases (injection, overflow)

Provide the complete test code with imports and setup:"""
        )
        
        return prompts
    
    def generate_tests(
        self,
        code: str,
        function_name: str,
        test_type: str = "unit",
        additional_context: str = ""
    ) -> str:
        """Generate tests for given code using RAG"""
        
        # Retrieve relevant context using RAG
        query = f"""Generate tests for {function_name}. 
        Code: {code[:500]}...
        Additional context: {additional_context}"""
        
        relevant_docs = self.vector_manager.retrieve_relevant_context(
            query=query,
            k=self.config.top_k_retrieval
        )
        
        # Format context from retrieved documents
        context = "\n\n---\n\n".join([
            f"File: {doc.metadata.get('file_path', 'unknown')}\n{doc.page_content}"
            for doc in relevant_docs
        ])
        
        # Generate tests using LLM
        prompt_template = self.prompts.get(test_type, self.prompts['unit'])
        prompt = prompt_template.format(
            context=context,
            code=code,
            function_name=function_name
        )
        
        console.print(f"[cyan]Generating {test_type} tests for {function_name}...[/cyan]")
        
        response = self.llm.invoke(prompt)
        generated_tests = response.content
        
        return generated_tests
    
    def generate_comprehensive_tests(
        self,
        code: str,
        function_name: str,
        test_types: List[str] = None
    ) -> Dict[str, str]:
        """Generate multiple types of tests"""
        if test_types is None:
            test_types = self.config.test_types
        
        all_tests = {}
        
        for test_type in test_types:
            tests = self.generate_tests(code, function_name, test_type)
            all_tests[test_type] = tests
        
        return all_tests

console.print("[bold green]Test case generator class defined[/bold green]")

## 6. Test Evaluation and Metrics

In [ ]:
class TestEvaluator:
    """Evaluate generated test cases"""
    
    def __init__(self):
        self.metrics = defaultdict(dict)
    
    def analyze_test_code(self, test_code: str) -> Dict[str, Any]:
        """Analyze test code quality"""
        metrics = {
            'num_test_functions': 0,
            'num_assertions': 0,
            'has_fixtures': False,
            'has_mocks': False,
            'has_parametrize': False,
            'lines_of_code': len(test_code.split('\n')),
        }
        
        try:
            # Parse test code
            tree = ast.parse(test_code)
            
            # Count test functions
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    if node.name.startswith('test_'):
                        metrics['num_test_functions'] += 1
                    
                    # Check for fixtures
                    if any(d.id == 'fixture' if isinstance(d, ast.Name) else False for d in node.decorator_list):
                        metrics['has_fixtures'] = True
                    
                    # Check for parametrize
                    for dec in node.decorator_list:
                        if isinstance(dec, ast.Call) and hasattr(dec.func, 'attr'):
                            if dec.func.attr == 'parametrize':
                                metrics['has_parametrize'] = True
                
                # Count assertions
                if isinstance(node, ast.Assert):
                    metrics['num_assertions'] += 1
                
                # Check for assert calls
                if isinstance(node, ast.Call):
                    if hasattr(node.func, 'id') and 'assert' in node.func.id.lower():
                        metrics['num_assertions'] += 1
            
            # Check for mocking
            if 'mock' in test_code.lower() or 'patch' in test_code.lower():
                metrics['has_mocks'] = True
                
        except SyntaxError as e:
            metrics['parse_error'] = str(e)
        
        return metrics
    
    def calculate_quality_score(self, metrics: Dict[str, Any]) -> float:
        """Calculate overall test quality score (0-100)"""
        score = 0
        
        # Test function count (up to 30 points)
        score += min(metrics.get('num_test_functions', 0) * 10, 30)
        
        # Assertion count (up to 20 points)
        score += min(metrics.get('num_assertions', 0) * 5, 20)
        
        # Best practices (50 points total)
        if metrics.get('has_fixtures'):
            score += 15
        if metrics.get('has_mocks'):
            score += 15
        if metrics.get('has_parametrize'):
            score += 20
        
        return min(score, 100)
    
    def evaluate_test_suite(
        self,
        test_suite: Dict[str, str],
        function_name: str
    ) -> Dict[str, Any]:
        """Evaluate a complete test suite"""
        evaluation = {
            'function_name': function_name,
            'test_types': {},
            'overall_metrics': {
                'total_tests': 0,
                'total_assertions': 0,
                'total_lines': 0,
            },
            'quality_scores': {}
        }
        
        for test_type, test_code in test_suite.items():
            metrics = self.analyze_test_code(test_code)
            score = self.calculate_quality_score(metrics)
            
            evaluation['test_types'][test_type] = metrics
            evaluation['quality_scores'][test_type] = score
            
            # Update overall metrics
            evaluation['overall_metrics']['total_tests'] += metrics.get('num_test_functions', 0)
            evaluation['overall_metrics']['total_assertions'] += metrics.get('num_assertions', 0)
            evaluation['overall_metrics']['total_lines'] += metrics.get('lines_of_code', 0)
        
        # Calculate average quality score
        if evaluation['quality_scores']:
            evaluation['overall_metrics']['average_quality'] = sum(evaluation['quality_scores'].values()) / len(evaluation['quality_scores'])
        
        return evaluation
    
    def print_evaluation_report(self, evaluation: Dict[str, Any]):
        """Print a formatted evaluation report"""
        console.print("\n[bold cyan]Test Evaluation Report[/bold cyan]")
        console.print(f"Function: [bold]{evaluation['function_name']}[/bold]\n")
        
        # Create table
        table = Table(show_header=True, header_style="bold magenta")
        table.add_column("Test Type", style="cyan")
        table.add_column("Tests", justify="right")
        table.add_column("Assertions", justify="right")
        table.add_column("Quality Score", justify="right")
        
        for test_type, metrics in evaluation['test_types'].items():
            score = evaluation['quality_scores'][test_type]
            score_color = "green" if score >= 70 else "yellow" if score >= 50 else "red"
            
            table.add_row(
                test_type,
                str(metrics.get('num_test_functions', 0)),
                str(metrics.get('num_assertions', 0)),
                f"[{score_color}]{score:.1f}/100[/{score_color}]"
            )
        
        console.print(table)
        
        # Print overall metrics
        console.print("\n[bold]Overall Metrics:[/bold]")
        console.print(f"  Total Tests: {evaluation['overall_metrics']['total_tests']}")
        console.print(f"  Total Assertions: {evaluation['overall_metrics']['total_assertions']}")
        console.print(f"  Total Lines: {evaluation['overall_metrics']['total_lines']}")
        console.print(f"  Average Quality: {evaluation['overall_metrics'].get('average_quality', 0):.1f}/100\n")

evaluator = TestEvaluator()
console.print("[bold green]Test evaluator initialized[/bold green]")

## 7. Complete RAG Test Generation Pipeline

In [ ]:
class RAGTestGenerationPipeline:
    """Complete end-to-end pipeline for RAG-based test generation"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.preprocessor = CodePreprocessor(config)
        self.vector_manager = VectorStoreManager(config)
        self.generator = None  # Will be initialized after vector store is ready
        self.evaluator = TestEvaluator()
    
    def index_codebase(self, code_directory: str, force_reindex: bool = False):
        """Index the codebase for RAG retrieval"""
        console.print(f"\n[bold cyan]Step 1: Indexing codebase from {code_directory}[/bold cyan]")
        
        # Check if vector store already exists
        if not force_reindex and os.path.exists(self.config.persist_directory):
            console.print("[yellow]Vector store exists. Loading...[/yellow]")
            self.vector_manager.load_vector_store()
            return
        
        # Load code files
        code_files = self.preprocessor.load_code_files(code_directory)
        console.print(f"[green]Found {len(code_files)} code files[/green]")
        
        if not code_files:
            console.print("[red]No code files found! Please check the directory.[/red]")
            return
        
        # Create documents
        documents = self.preprocessor.create_documents(code_files)
        console.print(f"[green]Created {len(documents)} documents[/green]")
        
        # Create vector store
        self.vector_manager.create_vector_store(documents)
        
        # Save vector store
        self.vector_manager.save_vector_store()
        
        console.print("[bold green]Codebase indexed successfully![/bold green]")
    
    def initialize_generator(self):
        """Initialize test generator after vector store is ready"""
        if self.vector_manager.vector_store is None:
            console.print("[red]Error: Vector store not initialized. Please index codebase first.[/red]")
            return False
        
        console.print("\n[bold cyan]Step 2: Initializing test generator[/bold cyan]")
        self.generator = TestCaseGenerator(self.config, self.vector_manager)
        console.print("[bold green]Test generator ready![/bold green]")
        return True
    
    def generate_tests_for_function(
        self,
        code: str,
        function_name: str,
        test_types: List[str] = None,
        save_to_file: bool = True,
        output_dir: str = "./generated_tests"
    ) -> Dict[str, str]:
        """Generate tests for a specific function"""
        if self.generator is None:
            if not self.initialize_generator():
                return {}
        
        console.print(f"\n[bold cyan]Step 3: Generating tests for '{function_name}'[/bold cyan]")
        
        # Generate tests
        test_suite = self.generator.generate_comprehensive_tests(
            code=code,
            function_name=function_name,
            test_types=test_types
        )
        
        # Evaluate tests
        console.print("\n[bold cyan]Step 4: Evaluating generated tests[/bold cyan]")
        evaluation = self.evaluator.evaluate_test_suite(test_suite, function_name)
        self.evaluator.print_evaluation_report(evaluation)
        
        # Save to file
        if save_to_file:
            os.makedirs(output_dir, exist_ok=True)
            for test_type, test_code in test_suite.items():
                filename = f"test_{function_name}_{test_type}.py"
                filepath = os.path.join(output_dir, filename)
                with open(filepath, 'w') as f:
                    f.write(test_code)
                console.print(f"[green]Saved {test_type} tests to {filepath}[/green]")
        
        return test_suite
    
    def generate_tests_for_file(
        self,
        file_path: str,
        output_dir: str = "./generated_tests"
    ) -> Dict[str, Dict[str, str]]:
        """Generate tests for all functions in a file"""
        with open(file_path, 'r') as f:
            code = f.read()
        
        # Parse functions
        parser = CodeParser()
        functions = parser.extract_functions_python(code, file_path)
        
        all_tests = {}
        
        for func in functions:
            func_name = func['name']
            func_source = func.get('source', '')
            
            if func_source:
                tests = self.generate_tests_for_function(
                    code=func_source,
                    function_name=func_name,
                    output_dir=output_dir
                )
                all_tests[func_name] = tests
        
        return all_tests

# Initialize the pipeline
pipeline = RAGTestGenerationPipeline(config)
console.print("[bold green]RAG Test Generation Pipeline initialized![/bold green]")

## 8. Example Usage - Demo Code

In [ ]:
# Create a sample code directory for demonstration
sample_code_dir = "./sample_code"
os.makedirs(sample_code_dir, exist_ok=True)

# Create sample Python file
sample_code = '''
"""Sample calculator module for demonstration"""

from typing import Union, List
import math

class Calculator:
    """A simple calculator class"""
    
    def __init__(self):
        self.history = []
    
    def add(self, a: float, b: float) -> float:
        """Add two numbers"""
        result = a + b
        self.history.append(f"{a} + {b} = {result}")
        return result
    
    def divide(self, a: float, b: float) -> float:
        """Divide two numbers
        
        Args:
            a: Numerator
            b: Denominator
            
        Returns:
            Result of division
            
        Raises:
            ZeroDivisionError: If b is zero
        """
        if b == 0:
            raise ZeroDivisionError("Cannot divide by zero")
        result = a / b
        self.history.append(f"{a} / {b} = {result}")
        return result
    
    def get_history(self) -> List[str]:
        """Return calculation history"""
        return self.history.copy()


def calculate_average(numbers: List[float]) -> float:
    """Calculate the average of a list of numbers
    
    Args:
        numbers: List of numbers to average
        
    Returns:
        The arithmetic mean
        
    Raises:
        ValueError: If list is empty
    """
    if not numbers:
        raise ValueError("Cannot calculate average of empty list")
    return sum(numbers) / len(numbers)


def factorial(n: int) -> int:
    """Calculate factorial of n
    
    Args:
        n: Non-negative integer
        
    Returns:
        n!
        
    Raises:
        ValueError: If n is negative
    """
    if n < 0:
        raise ValueError("Factorial not defined for negative numbers")
    if n == 0 or n == 1:
        return 1
    return n * factorial(n - 1)
'''

with open(f"{sample_code_dir}/calculator.py", 'w') as f:
    f.write(sample_code)

console.print("[green]Sample code created in ./sample_code/calculator.py[/green]")

## 9. Run the Pipeline

In [ ]:
# Step 1: Index the codebase
pipeline.index_codebase(sample_code_dir, force_reindex=True)

In [ ]:
# Step 2: Initialize the generator
pipeline.initialize_generator()

In [ ]:
# Step 3: Generate tests for a specific function
test_function_code = '''
def calculate_average(numbers: List[float]) -> float:
    """Calculate the average of a list of numbers
    
    Args:
        numbers: List of numbers to average
        
    Returns:
        The arithmetic mean
        
    Raises:
        ValueError: If list is empty
    """
    if not numbers:
        raise ValueError("Cannot calculate average of empty list")
    return sum(numbers) / len(numbers)
'''

# Generate comprehensive tests
generated_tests = pipeline.generate_tests_for_function(
    code=test_function_code,
    function_name="calculate_average",
    test_types=["unit", "edge_case"],
    save_to_file=True
)

In [ ]:
# View generated unit tests
console.print("\n[bold cyan]Generated Unit Tests:[/bold cyan]")
console.print(generated_tests.get('unit', 'No unit tests generated'))

In [ ]:
# View generated edge case tests
console.print("\n[bold cyan]Generated Edge Case Tests:[/bold cyan]")
console.print(generated_tests.get('edge_case', 'No edge case tests generated'))

## 10. Generate Tests for Entire File

In [ ]:
# Generate tests for all functions in the calculator.py file
all_tests = pipeline.generate_tests_for_file(
    file_path=f"{sample_code_dir}/calculator.py",
    output_dir="./generated_tests"
)

console.print(f"\n[bold green]Generated tests for {len(all_tests)} functions[/bold green]")

## 11. Interactive Test Generation

In [ ]:
def interactive_test_generation():
    """Interactive interface for test generation"""
    console.print("\n[bold cyan]Interactive Test Generation[/bold cyan]")
    console.print("Enter your code (type 'END' on a new line when done):\n")
    
    lines = []
    while True:
        line = input()
        if line.strip() == 'END':
            break
        lines.append(line)
    
    code = '\n'.join(lines)
    
    function_name = input("\nEnter function/class name: ")
    test_types_input = input("Enter test types (comma-separated, e.g., unit,edge_case,integration): ")
    
    test_types = [t.strip() for t in test_types_input.split(',')]
    
    # Generate tests
    tests = pipeline.generate_tests_for_function(
        code=code,
        function_name=function_name,
        test_types=test_types
    )
    
    return tests

# Uncomment to use interactive mode
# interactive_tests = interactive_test_generation()

## 12. Advanced Features

In [ ]:
# Search for relevant code examples
def search_codebase(query: str, k: int = 5):
    """Search the codebase for relevant examples"""
    results = pipeline.vector_manager.retrieve_relevant_context(query, k=k)
    
    console.print(f"\n[bold cyan]Search Results for: '{query}'[/bold cyan]\n")
    
    for i, doc in enumerate(results, 1):
        console.print(f"[bold]Result {i}:[/bold]")
        console.print(f"File: {doc.metadata.get('file_path', 'unknown')}")
        console.print(f"Type: {doc.metadata.get('type', 'unknown')}")
        console.print(f"Content:\n{doc.page_content[:200]}...\n")

# Example search
# search_codebase("calculator division error handling")

In [ ]:
# Batch test generation for multiple functions
def batch_generate_tests(functions_list: List[Dict[str, str]], output_dir: str = "./batch_tests"):
    """Generate tests for multiple functions in batch
    
    Args:
        functions_list: List of dicts with 'code' and 'name' keys
        output_dir: Directory to save generated tests
    """
    os.makedirs(output_dir, exist_ok=True)
    results = {}
    
    for func_info in tqdm(functions_list, desc="Generating tests"):
        tests = pipeline.generate_tests_for_function(
            code=func_info['code'],
            function_name=func_info['name'],
            save_to_file=True,
            output_dir=output_dir
        )
        results[func_info['name']] = tests
    
    return results

# Example batch generation
# batch_functions = [
#     {'name': 'function1', 'code': '...'},
#     {'name': 'function2', 'code': '...'}
# ]
# batch_results = batch_generate_tests(batch_functions)

## 13. Export and Reporting

In [ ]:
def generate_report(output_file: str = "test_generation_report.json"):
    """Generate a comprehensive report of test generation"""
    report = {
        'config': asdict(config),
        'timestamp': str(Path.ctime(Path('.'))),
        'statistics': {
            'total_functions_processed': 0,
            'total_tests_generated': 0,
            'average_quality_score': 0
        }
    }
    
    with open(output_file, 'w') as f:
        json.dump(report, f, indent=2)
    
    console.print(f"[green]Report saved to {output_file}[/green]")
    return report

# Generate report
# report = generate_report()

## 14. Fine-tuning Preparation (Optional)

In [ ]:
def prepare_finetuning_dataset(
    code_test_pairs: List[Dict[str, str]],
    output_file: str = "finetuning_dataset.jsonl"
):
    """Prepare dataset for LLM fine-tuning
    
    Args:
        code_test_pairs: List of dicts with 'code' and 'tests' keys
        output_file: Output JSONL file for fine-tuning
    """
    with open(output_file, 'w') as f:
        for pair in code_test_pairs:
            training_example = {
                "messages": [
                    {
                        "role": "system",
                        "content": "You are an expert software testing engineer. Generate comprehensive test cases."
                    },
                    {
                        "role": "user",
                        "content": f"Generate unit tests for the following code:\n\n{pair['code']}"
                    },
                    {
                        "role": "assistant",
                        "content": pair['tests']
                    }
                ]
            }
            f.write(json.dumps(training_example) + '\n')
    
    console.print(f"[green]Fine-tuning dataset saved to {output_file}[/green]")
    console.print(f"[yellow]Total examples: {len(code_test_pairs)}[/yellow]")

# Example usage
# training_pairs = [
#     {'code': '...', 'tests': '...'},
#     {'code': '...', 'tests': '...'}
# ]
# prepare_finetuning_dataset(training_pairs)

## 15. Summary and Next Steps

In [ ]:
console.print("""
[bold cyan]RAG-Based Test Generation System - Summary[/bold cyan]

[bold]What we built:[/bold]
✓ Code parser and preprocessor for extracting functions/classes
✓ Vector store with embeddings for semantic code search
✓ RAG-based retrieval system for relevant context
✓ LLM-powered test generation (unit, integration, edge cases)
✓ Test evaluation and quality metrics
✓ Complete end-to-end pipeline

[bold]How to use:[/bold]
1. Set your API keys (OPENAI_API_KEY or ANTHROPIC_API_KEY)
2. Index your codebase: pipeline.index_codebase("path/to/code")
3. Generate tests: pipeline.generate_tests_for_function(code, name)
4. Review generated tests in ./generated_tests/

[bold]Next steps:[/bold]
• Fine-tune LLM on your specific codebase
• Integrate with CI/CD pipeline
• Run generated tests and measure coverage
• Iterate based on test results
• Build feedback loop for continuous improvement

[bold]Advanced features:[/bold]
• Support for multiple languages (JS, Java, C++, etc.)
• Integration with pytest/unittest
• Coverage analysis and gap detection
• Automated test execution and validation
• Test maintenance and updates
""")